In [ ]:
import os
import glob
import numpy as np
import pandas as pd

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260905_123159"

def add_trade_impact_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Adds future return at multiple time horizons using snapshots.
    Assumes both inputs are already sorted by ts.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        out[f"trade_return_{h}ms"] = (
            (merged[f"future_mid_{h}"] - out["price"]) / out["price"]
        )

    return out

def label_toxicity_multi(trades: pd.DataFrame, snapshots: pd.DataFrame, horizons_ms=(100, 500, 1000, 5000)):
    """
    Toxicity = price moved against aggressor after trade.
    Multi-horizon version.
    Assumes sorted inputs.
    """
    out = trades.copy()

    for h in horizons_ms:

        future = snapshots[["ts", "mid"]].copy()
        future["ts"] = future["ts"] - h
        future = future.rename(columns={"mid": f"future_mid_{h}"})

        merged = pd.merge_asof(
            out,
            future,
            on="ts",
            direction="forward"
        )

        move = merged[f"future_mid_{h}"] - out["price"]

        out[f"toxicity_{h}ms"] = np.where(
            out["side"] == "BUY",
            move < 0,
            move > 0
        ).astype(int)

        out[f"signed_impact_{h}ms"] = np.where(
            out["side"] == "BUY",
            -move,
            move
        )

        out[f"signed_impact_bps_{h}ms"] = (
            10000 * out[f"signed_impact_{h}ms"] / out["price"]
        )

    return out

def align_trades_to_snapshots(trades: pd.DataFrame, snapshots: pd.DataFrame):

    snap = snapshots[["ts", "mid", "best_bid", "best_ask"]].rename(columns={
        "mid": "snap_mid",
        "best_bid": "snap_bid",
        "best_ask": "snap_ask"
    })

    merged = pd.merge_asof(trades, snap, on="ts", direction="backward")

    trades = merged.copy()

    trades["snapshot_mid"] = trades["snap_mid"]
    trades["snapshot_bid"] = trades["snap_bid"]
    trades["snapshot_ask"] = trades["snap_ask"]
    trades["spread"] = trades["snap_ask"] - trades["snap_bid"]

    return trades

def label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000)):

    snap_ts = snapshots["ts"].values
    snap_mid = snapshots["mid"].values

    fills = fills.copy()

    base_idx = np.searchsorted(snap_ts, fills["ts"].values, side="right") - 1
    base_idx = np.clip(base_idx, 0, len(snap_ts) - 1)

    fills["snap_mid"] = snap_mid[base_idx]

    fill_ts = fills["ts"].values
    fill_price = fills["price"].values
    
    for h in horizons_ms:
        target = fill_ts + h

        idx = np.searchsorted(snap_ts, target, side="left")
        idx = np.clip(idx, 0, len(snap_ts) - 1)

        # NEW CORRECT SIGNED LOGIC
        # Assume: +1 = buy, -1 = sell
        fills[f"signed_markout_{h}ms"] = fills["fill_sign"] * (snap_mid[idx] - fill_price)
        fills[f"signed_adverse_{h}ms"] = -fills["fill_sign"] * (snap_mid[idx] - fill_price)

    # NEW FOR TOXICITY MODELLING
    snapshots["micro_signal"] = (snapshots["microprice_dev"]) / snapshots["mid"]

    snapshot_cols = [
        "ts",
        "symbol",
        "mid",
        "microprice",
        "microprice_dev",
        "micro_signal",
        "spread",
        "volatility",
        "order_imbalance",
        "trade_imbalance",
        "quote_churn",
        "inventory",
        "regime",
    ]

    snap = snapshots[snapshot_cols].copy()
    snap = snap.rename(columns={"ts": "snapshot_ts"})
    fills = fills.rename(columns={"ts": "fill_ts"})

    fills = pd.merge_asof(
        fills,
        snap,
        left_on="fill_ts",
        right_on="snapshot_ts",
        by="symbol",
        direction="backward",
    )

    fills["snapshot_age_ms"] = fills["fill_ts"] - fills["snapshot_ts"]
    fills = fills.drop(columns=[col for col in fills.columns if col.endswith("_y")])
    fills.columns = fills.columns.str.replace("_x", "", regex=False)

    regime_dummies = pd.get_dummies(fills["regime_id"], prefix="regime").reindex( # one hot encoding for regime toxicity modelling
        columns=["regime_0", "regime_1", "regime_2"], fill_value=0).astype(np.float32)

    fills = pd.concat([fills, regime_dummies], axis=1)

    return fills

def forward_index(ts, horizon):
    n = len(ts)
    j = 0
    idx = np.empty(n, dtype=int)

    for i in range(n):
        while j < n and ts[j] < ts[i] + horizon:
            j += 1
        idx[i] = j if j < n else n

    return idx

def finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000)):

    ts = snapshots["ts"].to_numpy()
    mid = snapshots["mid"].to_numpy()
    n = len(snapshots)

    for h in horizons_ms:
        idx = forward_index(ts, h)

        future_mid = np.full(n, np.nan)
        valid = idx < n

        future_mid[valid] = mid[idx[valid]]
        
        # Store future mid
        snapshots[f"future_mid_{h}ms"] = future_mid

        # Store return
        snapshots[f"future_return_{h}ms"] = (future_mid - mid) / mid

    return snapshots

def compute_utility(snapshots, lambda_inv=1.0, gamma_tail=1.0):

    df = snapshots.copy()

    pnl = df["total_pnl"].values
    inventory = df["inventory"].values

    # 1. PnL flow (NOT level)
    pnl_flow = np.diff(pnl, prepend=pnl[0])

    pnl_term = np.mean(pnl_flow)

    # 2. Inventory risk (correct time-aligned)
    inv_risk = np.mean(inventory**2)

    # 3. Tail risk (on pnl flow, not level)
    losses = -pnl_flow

    var = np.quantile(losses, 0.95)
    cvar = losses[losses >= var].mean()

    utility = pnl_term - lambda_inv * inv_risk - gamma_tail * cvar

    print("PnL term:", pnl_term)
    print("Inventory penalty:", lambda_inv * inv_risk)
    print("Tail penalty:", gamma_tail * cvar)
    print("Utility:", utility)

    return utility

def generate_datasets():

    def chunk_id(path):
        return int(os.path.basename(path).split("_")[1].split(".")[0])

    events_files = sorted(glob.glob(os.path.join(folder_path, "events_*.parquet")), key = chunk_id)
    events = pd.concat((pd.read_parquet(f) for f in events_files), ignore_index=True)

    snapshots_files = sorted(glob.glob(os.path.join(folder_path, "snapshots_*.parquet")), key = chunk_id)
    snapshots = pd.concat((pd.read_parquet(f) for f in snapshots_files), ignore_index=True)

    trades_files = sorted(glob.glob(os.path.join(folder_path, "trades_*.parquet")), key = chunk_id)
    trades = pd.concat((pd.read_parquet(f) for f in trades_files), ignore_index=True)

    quotes_files = sorted(glob.glob(os.path.join(folder_path, "quotes_*.parquet")), key = chunk_id)
    quotes = pd.concat((pd.read_parquet(f) for f in quotes_files), ignore_index=True)

    fills_files = sorted(glob.glob(os.path.join(folder_path, "fills_*.parquet")), key = chunk_id)
    fills = pd.concat((pd.read_parquet(f) for f in fills_files), ignore_index=True)

    events = events.sort_values("local_ts").copy()
    # events = events.sort_values("ts").copy()
    snapshots = snapshots.sort_values("ts").copy()
    trades = trades.sort_values("ts").copy()
    quotes = quotes.sort_values("ts").copy()
    fills = fills.sort_values("ts").copy()

    # Delete chunk files
    for files in [events_files, snapshots_files, trades_files, quotes_files, fills_files]:
        for f in files:
            os.remove(f)

    fills = label_fill_markouts_ms(fills, snapshots, horizons_ms=(100, 500, 1000, 5000))
    
    trades = add_trade_impact_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = label_toxicity_multi(trades, snapshots, horizons_ms=(100, 500, 1000, 5000))
    trades = align_trades_to_snapshots(trades, snapshots)

    snapshots = finalize_returns(snapshots, horizons_ms=(100, 500, 1000, 5000))

    compute_utility(snapshots)

    events.to_parquet(os.path.join(folder_path, "events.parquet"), index=False)
    snapshots.to_parquet(os.path.join(folder_path, "snapshots.parquet"), index=False)
    trades.to_parquet(os.path.join(folder_path, "trades.parquet"), index=False)
    quotes.to_parquet(os.path.join(folder_path, "quotes.parquet"), index=False)
    fills.to_parquet(os.path.join(folder_path, "fills.parquet"), index=False)

generate_datasets()

PnL term: -1.7023974499187033e-05
Inventory penalty: 510075897736.91693
Tail penalty: 3.006299200685282e-05
Utility: -510075897736.91693
